# 🦁 Savanna Acoustic Threat Detection — RL Training on Kaggle
**Algorithms:** DQN · REINFORCE · PPO · A2C  
**Timesteps:** 100,000 per run  
**GPU:** Enabled (set in Kaggle sidebar → Accelerator → GPU T4 x2)

## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

packages = [
    'gymnasium==0.29.1',
    'stable-baselines3==2.2.1',
    'sb3-contrib==2.2.1',
    'pygame==2.5.2',
    'tqdm',
    'rich',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed.')

## Step 2 — Upload your project files
Upload the savanna_rl folder to Kaggle using the **+ Add Data** button, or paste the environment code directly below.

In [ ]:
import os, sys

# If you uploaded as a Kaggle dataset, adjust this path:
PROJECT_PATH = '/kaggle/input/savanna-rl'   # adjust to match your dataset name

# If files are not found, copy them to working dir
if not os.path.exists(PROJECT_PATH):
    PROJECT_PATH = '/kaggle/working'
    print('Using working directory — paste your files here')

sys.path.insert(0, PROJECT_PATH)
os.makedirs('/kaggle/working/models/dqn', exist_ok=True)
os.makedirs('/kaggle/working/models/pg',  exist_ok=True)
os.makedirs('/kaggle/working/results',    exist_ok=True)
os.makedirs('/kaggle/working/logs',       exist_ok=True)
print('Paths ready.')

## Step 3 — Memory management utilities

In [ ]:
import gc
import torch
import psutil

def clear_memory(label=''):
    """Free RAM and GPU memory between training runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    ram = psutil.virtual_memory()
    used_gb  = (ram.total - ram.available) / 1e9
    total_gb = ram.total / 1e9
    print(f'  [{label}] Memory cleared | RAM: {used_gb:.1f}/{total_gb:.1f} GB '
          f'({ram.percent:.0f}% used)')
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        print(f'  GPU VRAM allocated: {allocated:.2f} GB')

def check_memory(warn_threshold=80):
    """Warn if RAM usage is critically high."""
    ram = psutil.virtual_memory()
    if ram.percent > warn_threshold:
        print(f'WARNING: RAM at {ram.percent:.0f}% — clearing before next run')
        clear_memory('auto')

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
clear_memory('startup')

## Step 4 — Kaggle-compatible environment (no pygame display)

In [ ]:
# Disable pygame display for headless Kaggle environment
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

from environment.custom_env import SavannaAcousticEnv

# Quick smoke test
env = SavannaAcousticEnv(render_mode=None, seed=42)
obs, info = env.reset()
print(f'Observation shape : {obs.shape}')
print(f'Action space      : {env.action_space}')
obs2, reward, term, trunc, info = env.step(0)
print(f'Step reward       : {reward:.2f}')
env.close()
print('Environment OK.')

## Step 5 — Shared training utilities

In [ ]:
import json, time, numpy as np
from stable_baselines3.common.monitor    import Monitor
from stable_baselines3.common.env_util   import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks  import EvalCallback, BaseCallback

TOTAL_TIMESTEPS = 100_000   # full training run
RESULTS_DIR     = '/kaggle/working/results'
MODELS_DIR      = '/kaggle/working/models'

class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
        return True

def make_env_fn(seed=0):
    def _f():
        e = SavannaAcousticEnv(render_mode=None, seed=seed)
        return Monitor(e)
    return _f

def save_results(results, filename):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Saved: {path}')

print('Utilities ready.')

## Step 6 — Train DQN (10 hyperparameter runs)

In [ ]:
from stable_baselines3 import DQN

DQN_CONFIGS = [
    {'name':'baseline',    'lr':1e-4, 'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[128,128], 'notes':'SB3 defaults'},
    {'name':'high_lr',     'lr':5e-4, 'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[128,128], 'notes':'High LR'},
    {'name':'deep_net',    'lr':1e-4, 'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[256,256,128], 'notes':'Deeper network'},
    {'name':'large_buf',   'lr':1e-4, 'buf':200000, 'bs':128, 'tau':1.0,  'gamma':0.99,  'exp_fr':0.30, 'exp_fe':0.05, 'arch':[128,128], 'notes':'Large buffer'},
    {'name':'low_gamma',   'lr':1e-4, 'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.90,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[128,128], 'notes':'Low gamma'},
    {'name':'soft_update', 'lr':1e-4, 'buf':50000,  'bs':64,  'tau':0.01, 'gamma':0.99,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[128,128], 'notes':'Polyak update'},
    {'name':'long_explore','lr':1e-4, 'buf':100000, 'bs':64,  'tau':1.0,  'gamma':0.99,  'exp_fr':0.50, 'exp_fe':0.10, 'arch':[128,128], 'notes':'Long exploration'},
    {'name':'large_batch', 'lr':2e-4, 'buf':100000, 'bs':256, 'tau':1.0,  'gamma':0.99,  'exp_fr':0.20, 'exp_fe':0.05, 'arch':[256,256], 'notes':'Large batch'},
    {'name':'fast_target', 'lr':1e-4, 'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99,  'exp_fr':0.15, 'exp_fe':0.02, 'arch':[128,128], 'notes':'Fast target update'},
    {'name':'best_tuned',  'lr':2e-4, 'buf':100000, 'bs':128, 'tau':0.05, 'gamma':0.995, 'exp_fr':0.30, 'exp_fe':0.03, 'arch':[256,128,64], 'notes':'Best tuned'},
]

dqn_results = []

for i, cfg in enumerate(DQN_CONFIGS, 1):
    check_memory()
    print(f'\nDQN Run {i}/10 — {cfg["name"]} | {cfg["notes"]}')

    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env,
        best_model_save_path=f'{MODELS_DIR}/dqn/{cfg["name"]}',
        eval_freq=5000, n_eval_episodes=5, verbose=0)

    model = DQN('MlpPolicy', train_env,
        learning_rate=cfg['lr'],
        buffer_size=cfg['buf'],
        batch_size=cfg['bs'],
        tau=cfg['tau'],
        gamma=cfg['gamma'],
        exploration_fraction=cfg['exp_fr'],
        exploration_final_eps=cfg['exp_fe'],
        policy_kwargs={'net_arch': cfg['arch']},
        learning_starts=1000,
        train_freq=4,
        target_update_interval=1000,
        verbose=1, seed=i*7)

    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0

    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    model.save(f'{MODELS_DIR}/dqn/{cfg["name"]}/final_model')

    res = {'algorithm':'DQN','run_id':i,'config_name':cfg['name'],
           'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),
           'training_time_s':float(elapsed),'episode_rewards':logger.episode_rewards[-50:]}
    dqn_results.append(res)
    print(f'  Reward: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed:.0f}s')

    # Clean up between runs
    del model, train_env, eval_env, logger
    clear_memory(f'after DQN run {i}')

save_results(dqn_results, 'dqn_all_results.json')
dqn_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest DQN: {dqn_results[0]["config_name"]} — {dqn_results[0]["mean_reward"]:.2f}')

## Step 7 — Train PPO (10 runs)

In [ ]:
from stable_baselines3 import PPO
clear_memory('before PPO')

PPO_CONFIGS = [
    {'name':'ppo_baseline',  'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],'notes':'SB3 defaults'},
    {'name':'ppo_sm_steps',  'lr':3e-4,'n_steps':512, 'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],'notes':'Small rollout'},
    {'name':'ppo_lg_steps',  'lr':3e-4,'n_steps':4096,'bs':128,'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[256,256],'notes':'Large rollout'},
    {'name':'ppo_hi_clip',   'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.4,'ent':0.00,'vf':0.5,'arch':[128,128],'notes':'High clip'},
    {'name':'ppo_lo_clip',   'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.1,'ent':0.00,'vf':0.5,'arch':[128,128],'notes':'Low clip'},
    {'name':'ppo_entropy',   'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[128,128],'notes':'Entropy bonus'},
    {'name':'ppo_high_vf',   'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':1.0,'arch':[128,128],'notes':'High value coeff'},
    {'name':'ppo_low_gae',   'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.80,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],'notes':'Low GAE lambda'},
    {'name':'ppo_more_ep',   'lr':2e-4,'n_steps':2048,'bs':64, 'epochs':20,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[256,128],'notes':'More SGD epochs'},
    {'name':'ppo_best',      'lr':2e-4,'n_steps':2048,'bs':128,'epochs':15,'gamma':0.995,'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[256,128,64],'notes':'Best tuned PPO'},
]

ppo_results = []

for i, cfg in enumerate(PPO_CONFIGS, 1):
    check_memory()
    print(f'\nPPO Run {i}/10 — {cfg["name"]} | {cfg["notes"]}')

    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env,
        best_model_save_path=f'{MODELS_DIR}/pg/ppo/{cfg["name"]}',
        eval_freq=5000, n_eval_episodes=5, verbose=0)

    model = PPO('MlpPolicy', train_env,
        learning_rate=cfg['lr'], n_steps=cfg['n_steps'],
        batch_size=cfg['bs'], n_epochs=cfg['epochs'],
        gamma=cfg['gamma'], gae_lambda=cfg['gae'],
        clip_range=cfg['clip'], ent_coef=cfg['ent'],
        vf_coef=cfg['vf'],
        policy_kwargs={'net_arch': cfg['arch']},
        verbose=1, seed=i*7)

    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0

    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    os.makedirs(f'{MODELS_DIR}/pg/ppo/{cfg["name"]}', exist_ok=True)
    model.save(f'{MODELS_DIR}/pg/ppo/{cfg["name"]}/final_model')

    res = {'algorithm':'PPO','run_id':i,'config_name':cfg['name'],
           'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),
           'training_time_s':float(elapsed),'episode_rewards':logger.episode_rewards[-50:]}
    ppo_results.append(res)
    print(f'  Reward: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed:.0f}s')

    del model, train_env, eval_env, logger
    clear_memory(f'after PPO run {i}')

save_results(ppo_results, 'ppo_all_results.json')
ppo_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest PPO: {ppo_results[0]["config_name"]} — {ppo_results[0]["mean_reward"]:.2f}')

## Step 8 — Train A2C (10 runs)

In [ ]:
from stable_baselines3 import A2C
clear_memory('before A2C')

A2C_CONFIGS = [
    {'name':'a2c_baseline', 'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[64,64],     'notes':'SB3 defaults'},
    {'name':'a2c_steps20',  'lr':7e-4,'n_steps':20,'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],   'notes':'Longer rollouts'},
    {'name':'a2c_high_lr',  'lr':1e-3,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],   'notes':'High LR'},
    {'name':'a2c_gae',      'lr':7e-4,'n_steps':10,'gamma':0.99, 'gae':0.95,'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],   'notes':'GAE estimation'},
    {'name':'a2c_entropy',  'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[128,128],   'notes':'Entropy bonus'},
    {'name':'a2c_high_vf',  'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':1.0,'gnorm':0.5,'arch':[128,128],   'notes':'High VF coeff'},
    {'name':'a2c_low_gam',  'lr':7e-4,'n_steps':5, 'gamma':0.90, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],   'notes':'Low gamma'},
    {'name':'a2c_deep',     'lr':5e-4,'n_steps':10,'gamma':0.99, 'gae':0.95,'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[256,256,128],'notes':'Deep network'},
    {'name':'a2c_gradclip', 'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.1,'arch':[128,128],   'notes':'Aggressive grad clip'},
    {'name':'a2c_best',     'lr':5e-4,'n_steps':20,'gamma':0.995,'gae':0.95,'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[256,128],   'notes':'Best tuned A2C'},
]

a2c_results = []

for i, cfg in enumerate(A2C_CONFIGS, 1):
    check_memory()
    print(f'\nA2C Run {i}/10 — {cfg["name"]} | {cfg["notes"]}')

    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env,
        best_model_save_path=f'{MODELS_DIR}/pg/a2c/{cfg["name"]}',
        eval_freq=5000, n_eval_episodes=5, verbose=0)

    model = A2C('MlpPolicy', train_env,
        learning_rate=cfg['lr'], n_steps=cfg['n_steps'],
        gamma=cfg['gamma'], gae_lambda=cfg['gae'],
        ent_coef=cfg['ent'], vf_coef=cfg['vf'],
        max_grad_norm=cfg['gnorm'],
        policy_kwargs={'net_arch': cfg['arch']},
        verbose=1, seed=i*7)

    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0

    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    os.makedirs(f'{MODELS_DIR}/pg/a2c/{cfg["name"]}', exist_ok=True)
    model.save(f'{MODELS_DIR}/pg/a2c/{cfg["name"]}/final_model')

    res = {'algorithm':'A2C','run_id':i,'config_name':cfg['name'],
           'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),
           'training_time_s':float(elapsed),'episode_rewards':logger.episode_rewards[-50:]}
    a2c_results.append(res)
    print(f'  Reward: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed:.0f}s')

    del model, train_env, eval_env, logger
    clear_memory(f'after A2C run {i}')

save_results(a2c_results, 'a2c_all_results.json')
a2c_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest A2C: {a2c_results[0]["config_name"]} — {a2c_results[0]["mean_reward"]:.2f}')

## Step 9 — Train REINFORCE (10 runs)

In [ ]:
clear_memory('before REINFORCE')
import torch, torch.nn as nn

class REINFORCEPolicy(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden=(128,128)):
        super().__init__()
        layers, in_d = [], obs_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.ReLU()]
            in_d = h
        layers.append(nn.Linear(in_d, action_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)
    def get_action(self, obs_t):
        logits = self.forward(obs_t)
        dist   = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        return a, dist.log_prob(a)
    def evaluate(self, obs_t, actions):
        dist = torch.distributions.Categorical(logits=self.forward(obs_t))
        return dist.log_prob(actions), dist.entropy()

def train_reinforce(cfg, run_id, total_timesteps):
    env    = Monitor(SavannaAcousticEnv(render_mode=None, seed=run_id*100))
    e_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    obs_d  = env.observation_space.shape[0]
    act_d  = env.action_space.n
    policy = REINFORCEPolicy(obs_d, act_d, cfg['arch'])
    opt    = torch.optim.Adam(policy.parameters(), lr=cfg['lr'])
    ep_rewards, total_steps = [], 0

    while total_steps < total_timesteps:
        obs, _ = env.reset()
        states, actions, rewards, log_probs = [], [], [], []
        done = False
        while not done:
            obs_t  = torch.FloatTensor(obs).unsqueeze(0)
            a, lp  = policy.get_action(obs_t)
            obs, r, term, trunc, _ = env.step(a.item())
            done = term or trunc
            states.append(obs_t); actions.append(a)
            rewards.append(r);    log_probs.append(lp)
            total_steps += 1

        ep_rewards.append(sum(rewards))
        G, returns = 0.0, []
        for r in reversed(rewards):
            G = r + cfg['gamma'] * G
            returns.insert(0, G)
        ret_t = torch.FloatTensor(returns)
        if cfg['baseline']:
            ret_t = (ret_t - ret_t.mean()) / (ret_t.std() + 1e-8)
        lp_t = torch.stack(log_probs)
        _, entropy = policy.evaluate(torch.cat(states), torch.stack(actions))
        loss = -(lp_t * ret_t).mean() - cfg['ent_coef'] * entropy.mean()
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
        opt.step()
        if len(ep_rewards) % 20 == 0:
            avg = np.mean(ep_rewards[-20:])
            print(f'    Step {total_steps:>7} | Ep {len(ep_rewards):>4} | Avg(20): {avg:.2f}')

    # Evaluate
    eval_rewards = []
    for _ in range(10):
        obs, _ = e_env.reset()
        ep_r, done = 0.0, False
        while not done:
            obs_t = torch.FloatTensor(obs).unsqueeze(0)
            with torch.no_grad():
                a, _ = policy.get_action(obs_t)
            obs, r, term, trunc, _ = e_env.step(a.item())
            ep_r += r; done = term or trunc
        eval_rewards.append(ep_r)

    save_dir = f'{MODELS_DIR}/pg/reinforce/{cfg["name"]}'
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'policy_state': policy.state_dict(), 'config': cfg},
               f'{save_dir}/final_model.pt')
    env.close(); e_env.close()
    return float(np.mean(eval_rewards)), float(np.std(eval_rewards)), ep_rewards


RF_CONFIGS = [
    {'name':'rf_baseline',    'lr':1e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),'notes':'Standard REINFORCE+baseline'},
    {'name':'rf_no_baseline', 'lr':1e-3, 'gamma':0.99, 'baseline':False,'ent_coef':0.01,'arch':(128,128),'notes':'No baseline (high variance)'},
    {'name':'rf_high_lr',     'lr':5e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),'notes':'High LR'},
    {'name':'rf_low_lr',      'lr':1e-4, 'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),'notes':'Low LR'},
    {'name':'rf_low_gamma',   'lr':1e-3, 'gamma':0.90, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),'notes':'Low gamma'},
    {'name':'rf_deep',        'lr':1e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(256,256,128),'notes':'Deep network'},
    {'name':'rf_hi_entropy',  'lr':1e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.05,'arch':(128,128),'notes':'High entropy'},
    {'name':'rf_no_entropy',  'lr':1e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.00,'arch':(128,128),'notes':'No entropy'},
    {'name':'rf_wide',        'lr':2e-3, 'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(512,256),'notes':'Wide network'},
    {'name':'rf_best',        'lr':2e-3, 'gamma':0.995,'baseline':True, 'ent_coef':0.02,'arch':(256,128),'notes':'Best tuned REINFORCE'},
]

rf_results = []

for i, cfg in enumerate(RF_CONFIGS, 1):
    check_memory()
    print(f'\nREINFORCE Run {i}/10 — {cfg["name"]} | {cfg["notes"]}')
    t0 = time.time()
    mean_r, std_r, ep_rews = train_reinforce(cfg, i, TOTAL_TIMESTEPS)
    elapsed = time.time() - t0
    res = {'algorithm':'REINFORCE','run_id':i,'config_name':cfg['name'],
           'notes':cfg['notes'],'mean_reward':mean_r,'std_reward':std_r,
           'training_time_s':elapsed,'episode_rewards':ep_rews[-50:]}
    rf_results.append(res)
    print(f'  Reward: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed:.0f}s')
    clear_memory(f'after RF run {i}')

save_results(rf_results, 'reinforce_all_results.json')
rf_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest REINFORCE: {rf_results[0]["config_name"]} — {rf_results[0]["mean_reward"]:.2f}')

## Step 10 — Cross-algorithm comparison

In [ ]:
import matplotlib.pyplot as plt

all_bests = [
    dqn_results[0], ppo_results[0], a2c_results[0], rf_results[0]
]

print('='*65)
print('  CROSS-ALGORITHM COMPARISON')
print('='*65)
print(f'  {"Algorithm":<15} {"Best Config":<22} {"Mean Reward":>12} {"Std":>8}')
print('  ' + '-'*58)
for r in all_bests:
    print(f'  {r["algorithm"]:<15} {r["config_name"]:<22} '
          f'{r["mean_reward"]:>12.2f} {r["std_reward"]:>8.2f}')
print('='*65)

# Bar chart
fig, ax = plt.subplots(figsize=(10,5))
algos   = [r['algorithm'] for r in all_bests]
rewards = [r['mean_reward'] for r in all_bests]
stds    = [r['std_reward']  for r in all_bests]
colors  = ['#3B82F6','#10B981','#F59E0B','#8B5CF6']
bars = ax.bar(algos, rewards, yerr=stds, color=colors, alpha=0.85,
              edgecolor='white', linewidth=1.2, capsize=6)
ax.set_title('Best reward per algorithm (100k timesteps)', fontsize=14, pad=12)
ax.set_ylabel('Mean Reward (10 eval episodes)')
ax.axhline(0, color='white', linewidth=0.5, alpha=0.3)
ax.set_facecolor('#0F172A')
fig.patch.set_facecolor('#0F172A')
ax.tick_params(colors='white'); ax.yaxis.label.set_color('white')
ax.title.set_color('white')
for spine in ax.spines.values():
    spine.set_color('#334155')
plt.tight_layout()
plt.savefig('/kaggle/working/results/algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## Step 11 — Download trained models
Go to **Kaggle sidebar → Output** and download the `/kaggle/working/` folder.

Or zip it for easy download:

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/savanna_rl_models', 'zip', '/kaggle/working/models')
shutil.make_archive('/kaggle/working/savanna_rl_results','zip', '/kaggle/working/results')
print('Zipped! Download from Kaggle Output tab.')